# [2장 3강] 실습: 모델 학습 시 발생 문제와 처리 방법

## 실습 목표

- 학습·테스트 성능을 비교하여 일반화 상태를 해석할 수 있다.
- Bias-Variance Trade-off 관점에서 Underfitting과 Overfitting을 설명할 수 있다.
- Ridge와 Lasso로 L2·L1 Regularization을 적용할 수 있다.
- 모델별 Weight, MAE와 RMSE를 비교할 수 있다.
- 심화 과정에서 PyTorch Loss에 L1·L2 Penalty를 직접 추가할 수 있다.

## 사용 데이터

- 파일: `insurance(2).csv`
- Feature: `age`, `sex`, `bmi`, `children`, `smoker`, `region`
- Label: `charges`

## 진행 방식

- 문제 설명과 요구사항을 확인한 뒤 코드 셀을 작성합니다.
- 실행 결과를 근거로 문제에 제시된 질문에 답합니다.

## 실습 준비: 데이터 불러오기

Google Colab에서는 파일을 Google Drive의 `MyDrive`에 저장합니다. Jupyter Notebook에서는 노트북과 같은 폴더에 저장합니다.

In [2]:
import pandas as pd
import numpy as np

try:
    from google.colab import drive
    drive.mount('/content/drive')
    file_path = '/content/drive/MyDrive/insurance.csv'
except ModuleNotFoundError:
    file_path = r'C:\Machine_Learning_worksheet\insurance.csv'

insurance_df = pd.read_csv(file_path)
display(insurance_df.head())
print('데이터 크기:', insurance_df.shape)
insurance_df.info()

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


데이터 크기: (1338, 7)
<class 'pandas.DataFrame'>
RangeIndex: 1338 entries, 0 to 1337
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       1338 non-null   int64  
 1   sex       1338 non-null   str    
 2   bmi       1338 non-null   float64
 3   children  1338 non-null   int64  
 4   smoker    1338 non-null   str    
 5   region    1338 non-null   str    
 6   charges   1338 non-null   float64
dtypes: float64(2), int64(2), str(3)
memory usage: 73.3 KB


## 필수 1: 기본 선형회귀 모델의 일반화 성능 확인

### 문제 1-1: 기준 모델의 학습·테스트 성능 비교하기

#### 문제 설명

학습 데이터에서 잘 맞는 모델이 새로운 고객에게도 비슷한 성능을 내는지 확인하기 위해 학습 데이터와 테스트 데이터의 MAE와 RMSE를 비교합니다.

#### 요구사항

1. 중복 행을 제거합니다.
2. `sex`, `smoker`, `region`에 첫 번째 범주를 제외한 One-Hot Encoding을 적용합니다.
3. X와 y를 분리하고 `random_state=42`로 학습 80%, 테스트 20%로 나눕니다.
4. `age`, `bmi`, `children`에 StandardScaler를 적용합니다. 학습 데이터에는 `fit_transform()`, 테스트 데이터에는 `transform()`을 사용합니다.
5. LinearRegression 모델을 학습합니다.
6. 학습·테스트 MAE와 RMSE 및 RMSE 차이의 절댓값을 출력합니다.
7. 결과를 Underfitting, Overfitting과 일반화 관점에서 설명합니다.

#### 결과 해석 작성

> 학습 RMSE와 테스트 RMSE의 차이를 근거로 심각한 Overfitting이 나타났다고 볼 수 있나요? 또한 이 차이만으로 Underfitting 여부까지 확정할 수 있나요?

## 필수 2: Ridge를 이용한 L2 정규화

### 문제 2-1: Ridge 모델 학습 및 비교하기

#### 문제 설명

Ridge를 적용하여 Weight가 지나치게 커지는 것을 제한하고 기본 선형회귀와 성능 및 Weight 크기를 비교합니다.

#### 요구사항

1. `Ridge(alpha=1.0)` 모델을 학습합니다.
2. 학습·테스트 MAE와 RMSE를 계산합니다.
3. LinearRegression과 Ridge의 Weight 절댓값 합을 계산합니다.
4. Feature별 Weight를 비교합니다.
5. Ridge 적용 후 Weight와 테스트 성능 변화를 설명합니다.

#### 결과 해석 작성

> Ridge 적용 후 Weight 절댓값 합과 테스트 RMSE는 어떻게 변했으며, 이 결과만으로 Ridge가 더 좋은 모델이라고 할 수 있나요?

In [4]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error

# --- 데이터 로드 & 전처리 (앞 셀 내용 재현) ---
file_path = r'C:\Machine_Learning_worksheet\insurance.csv'
insurance_df = pd.read_csv(file_path).drop_duplicates().reset_index(drop=True)

insurance_encoded = pd.get_dummies(
    insurance_df, columns=['sex', 'smoker', 'region'], drop_first=True
)
X = insurance_encoded.drop(columns='charges')
y = insurance_encoded['charges']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

num_cols = ['age', 'bmi', 'children']
scaler = StandardScaler()
X_train = X_train.copy()
X_test = X_test.copy()
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])

# --- 여기서부터 Ridge 비교 (기존 코드) ---
lr = LinearRegression()
lr.fit(X_train, y_train)
# ... 이하 동일

# LinearRegression이 앞 셀에 없다면 대비해 다시 학습
lr = LinearRegression()
lr.fit(X_train, y_train)

# 1. Ridge(alpha=1.0) 모델 학습
ridge = Ridge(alpha=1.0)
ridge.fit(X_train, y_train)

# 2. 학습·테스트 MAE, RMSE 계산
def evaluate(m, X, y):
    pred = m.predict(X)
    mae = mean_absolute_error(y, pred)
    rmse = np.sqrt(mean_squared_error(y, pred))
    return mae, rmse

lr_train_mae, lr_train_rmse = evaluate(lr, X_train, y_train)
lr_test_mae,  lr_test_rmse  = evaluate(lr, X_test,  y_test)
ridge_train_mae, ridge_train_rmse = evaluate(ridge, X_train, y_train)
ridge_test_mae,  ridge_test_rmse  = evaluate(ridge, X_test,  y_test)

print('--- LinearRegression ---')
print(f'학습 MAE: {lr_train_mae:.4f} | 학습 RMSE: {lr_train_rmse:.4f}')
print(f'테스트 MAE: {lr_test_mae:.4f} | 테스트 RMSE: {lr_test_rmse:.4f}')

print('\n--- Ridge (alpha=1.0) ---')
print(f'학습 MAE: {ridge_train_mae:.4f} | 학습 RMSE: {ridge_train_rmse:.4f}')
print(f'테스트 MAE: {ridge_test_mae:.4f} | 테스트 RMSE: {ridge_test_rmse:.4f}')

# 3. Weight 절댓값 합 계산
lr_weight_sum = np.sum(np.abs(lr.coef_))
ridge_weight_sum = np.sum(np.abs(ridge.coef_))
print('\n--- Weight 절댓값 합 ---')
print(f'LinearRegression: {lr_weight_sum:.4f}')
print(f'Ridge:            {ridge_weight_sum:.4f}')
print(f'차이(감소량):      {lr_weight_sum - ridge_weight_sum:.4f}')

# 4. Feature별 Weight 비교
print('\n--- Feature별 Weight 비교 ---')
comparison = pd.DataFrame({
    'Feature': X_train.columns,
    'LinearRegression': lr.coef_,
    'Ridge': ridge.coef_,
    '차이': lr.coef_ - ridge.coef_
})
print(comparison.to_string(index=False))


--- LinearRegression ---
학습 MAE: 4181.9015 | 학습 RMSE: 6081.1069
테스트 MAE: 4177.0456 | 테스트 RMSE: 5956.3429

--- Ridge (alpha=1.0) ---
학습 MAE: 4190.3632 | 학습 RMSE: 6081.3521
테스트 MAE: 4193.8919 | 테스트 RMSE: 5971.8617

--- Weight 절댓값 합 ---
LinearRegression: 31106.4325
Ridge:            30923.1201
차이(감소량):      183.3124

--- Feature별 Weight 비교 ---
         Feature  LinearRegression        Ridge         차이
             age       3472.975553  3468.386672   4.588882
             bmi       1927.828251  1923.174554   4.653697
        children        636.501185   637.333571  -0.832386
        sex_male       -101.542054   -91.826900  -9.715154
      smoker_yes      23077.764593 22940.793421 136.971172
region_northwest       -391.761455  -389.109069  -2.652386
region_southeast       -838.919616  -819.817531 -19.102084
region_southwest       -659.139752  -652.678382  -6.461369


## 필수 3: Lasso를 이용한 L1 정규화

### 문제 3-1: Lasso 모델 학습 및 0인 Weight 확인하기

#### 문제 설명

Lasso를 적용하여 일부 Weight가 0이 되는지 확인하고 Feature Selection과 유사한 효과를 해석합니다.

#### 요구사항

1. `Lasso(alpha=100.0, max_iter=10000)` 모델을 학습합니다.
2. 학습·테스트 MAE와 RMSE를 계산합니다.
3. 각 Feature의 Weight를 출력합니다.
4. Weight가 0인 Feature와 개수를 출력합니다.
5. L1 정규화가 Feature Selection과 유사한 이유를 설명합니다.

#### 결과 해석 작성

> Weight가 0이 된 Feature는 무엇이며, Lasso가 Feature Selection과 유사한 효과를 갖는 이유는 무엇인가요?

## 필수 4: LinearRegression, Ridge, Lasso 비교

### 문제 4-1: 모델별 성능과 Weight 비교하기

#### 문제 설명

세 모델의 학습·테스트 성능과 Weight를 하나의 표로 정리하여 정규화가 Bias와 Variance에 미치는 영향을 해석합니다.

#### 요구사항

1. 세 모델의 학습·테스트 MAE와 RMSE를 비교합니다.
2. Feature별 Weight 비교표를 만듭니다.
3. 모델별 Weight 절댓값 합과 0인 Weight 개수를 비교합니다.
4. Weight 절댓값이 가장 큰 Feature를 확인합니다.
5. Bias-Variance Trade-off 관점에서 결과를 설명합니다.

#### 결과 해석 작성

> 정규화 강도가 증가하면 Bias와 Variance는 일반적으로 어떻게 변하며, 현재 결과에서는 어떤 모델을 선택하는 것이 합리적인가요?

## 심화 1: PyTorch에서 L1·L2 정규화 직접 적용

### 문제 5-1: PyTorch L1·L2 모델 학습하기

#### 문제 설명

2장 2강의 PyTorch Gradient Descent를 확장하여 MSE Loss에 L1 또는 L2 Penalty를 직접 더합니다.

#### 요구사항

1. 데이터를 float32 Tensor로 변환하고 Seed를 42로 고정합니다.
2. L1과 L2 모델을 같은 초기값으로 생성합니다.
3. `l1_lambda=200.0`, `l2_lambda=1.0 / 학습 데이터 수`를 사용합니다.
4. L1은 Weight 절댓값 합, L2는 Weight 제곱합을 MSE에 더합니다. Bias는 Penalty에서 제외합니다.
5. SGD, Learning Rate 0.01로 각 모델을 2,000 Epoch 학습하고 400 Epoch마다 Loss를 출력합니다.
6. 테스트 MAE와 RMSE 및 Weight를 비교합니다.
7. L1·L2 Penalty가 Weight에 미치는 차이를 설명합니다.

#### 결과 해석 작성

> PyTorch Loss에 L1 또는 L2 Penalty를 추가하면 Weight 업데이트에 어떤 변화가 생기나요?

## 실습 마무리

1. 학습 성능만으로 모델을 선택하면 안 되는 이유는 무엇인가요?
2. Ridge와 Lasso의 Weight 변화는 어떻게 다른가요?
3. 정규화 강도가 너무 크면 어떤 문제가 발생할 수 있나요?